<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>📈Crypto</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if kaggle.json is stored in Google Drive

Mounted at /content/drive


In [ ]:
# !pip -q install tensorflow==2.8 >> log
# !apt -q install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 >> log
# !pip -q install -U tfds-nightly tensorflow_addons tensorflow >> log
# !pip -q install -U tensorflow_addons >> log  # update tfa in case students need to use it

In [ ]:
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 260330-Crypto   # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: 260330-Crypto
100% 5.94M/5.94M [00:00<00:00, 125MB/s]
Using competition: 260330-Crypto
  teamId  teamName                       submissionDate              score         
--------  -----------------------------  --------------------------  ------------  
15561573  6_NitishaRoseStevenCharles     2026-04-12 19:08:28.390000  0.6047725409  
15520986  1_abalmasov_appel_ho_bajaj     2026-04-13 00:40:56.286000  0.5843152516  
15562224  Team7_Ivan_Quinnan_Omid        2026-04-12 17:20:23.090000  0.5566319821  
15565980  Jeremy Cahill                  2026-04-12 19:50:36.216000  0.5074622472  
15497296  Cody Moxam                     2026-04-09 13:24:33.096000  0.4817836068  
15586968  Gaven Larson                   2026-04-13 00:17:42.620000  0.4817836068  
15597508  Team2_AlbertAndrewGurleenEvan  2026-04-13 01:36:27.083000  0.4611676157  
15572291  Group 3                        2026-04-12 22:13:34.243000  0.

See [more](https://nvidia.custhelp.com/app/answers/detail/a_id/3751/~/useful-nvidia-smi-queries) about NVIDIA GPU stats. Test your code in (free) Colab. It uses Tesla K80 GPU.

In [ ]:
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free,memory.used --format=csv

name, memory.total [MiB], memory.free [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 14913 MiB, 0 MiB


In [ ]:
%%time
%%capture
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os
import tensorflow as tf, tensorflow.keras as keras, tensorflow_datasets as tfds # tensorflow_addons as tfa
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout, MaxPooling2D, Conv2D, GlobalAveragePooling2D
from tensorflow.keras.layers import SimpleRNN, Flatten, Dense, RNN, LSTM, TimeDistributed
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60*5): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=5, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 6.57 s, sys: 1.37 s, total: 7.94 s
Wall time: 6.57 s


Your training data are 7 descriptive features for past 500K observations. See helpful [Tutorial to the G-Research Crypto Competition](https://www.kaggle.com/cstein06/tutorial-to-the-g-research-crypto-competition).

In [ ]:
tXY = pd.read_csv('tXY.csv', index_col='id'); tXY

,Count,Open,High,Low,Close,Volume,VWAP
id,,,,,,,
0,64,0.20,0.20,0.20,0.20,447,0.20
1,72,0.20,0.20,0.20,0.20,592,0.20
...,...,...,...,...,...,...,...
499998,1636,1.15,1.16,1.15,1.15,2615,1.15
499999,3228,1.13,1.14,1.12,1.13,3354,1.13


Your task is to forecast the closing price for all future time steps (index IDs below).

In [ ]:
pY = pd.read_csv('sampleSubmission.csv', index_col='id'); pY.T

id,500000,500001,500002,500003,500004,500005,500006,500007,500008,500009,...,524421,524422,524423,524424,524425,524426,524427,524428,524429,524430
Close,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 300 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Split observations into blocks of temporal train and test sets</b><h3>

<font color=green>This is your baseline DNN model. Remember to [seed all your experiments](https://keras.io/getting_started/faq/#how-can-i-obtain-reproducible-results-using-keras-during-development) for reproducibility. [Status of GPU-Determinism in TF](https://github.com/NVIDIA/framework-determinism/blob/master/tensorflow_status.md).

In [ ]:
# K, (N, p), Nx, Ny = 50, tXY.shape, 20000, len(pY)  # samples, dataset dim, train set size, forecast set size
# LtX, LtY = [], []
# for i in range(N-Ny-K, N-Ny):                     # populate K samples with past X series and future Y series
#   LtX.append(tXY.iloc[(i-Nx):i, :].values)        # X: historical 7Dim observations for Nx steps behind
#   LtY.append(tXY.loc[i:(i+Ny-1),'Close'].values)  # Y: future closing prices for Ny steps ahead
# taX, taY = np.array(LtX), np.array(LtY)           # training arrays past input X and future output Y
# print(f'taX.shape=(K,Nx,p)={taX.shape}; taY=(K,Ny)={taY.shape}')  # convert to 3-tensors

# df = pd.DataFrame(np.r_[taX[0,:,4], taY[0,:]], columns=['train future price'])
# ax = df.plot(figsize=(30,3), title=f'Training series (before and after)- just a closing price series (out of K={K})');
# pd.DataFrame(taX[0,:,4], columns=['train past price']).plot(grid=True, ax=ax);

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense

In [ ]:
df = tXY.copy()


df['delta'] = df['Close'].diff()


# BASIC FEATURES
df['OC_diff'] = df['Open'] - df['Close']
df['HL_diff'] = df['High'] - df['Low']


# RETURNS
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))

# ROLLING FEATURES
windows = [5, 10, 20]

for w in windows:
    df[f'roll_std_{w}'] = df['Close'].rolling(w).std()
    df[f'momentum_{w}'] = df['Close'] - df['Close'].shift(w)

    # slope
    df[f'slope_{w}'] = df['Close'].rolling(w).apply(
        lambda x: np.polyfit(range(len(x)), x, 1)[0], raw=True
    )


# DIRECTION
df['direction'] = (df['delta'] > 0).astype(int)

df = df.dropna()

<font color=green><h3><b>$\beta$. Build and fit RNN</b><h3>

<font color=green>Build an LSTM model with two hidden layers. It splits $K$ samples into batches with 7D series $X_{N_x\times p}$ as input and 1D series $Y_{N_y\times 1}$ as output.


In [ ]:
%%time
# tf.random.set_seed(0)   # always seed your experiments
# Init = keras.initializers.GlorotUniform(seed=0) # seed all that you can

# m = Sequential(
#   [LSTM(100, return_sequences=True, input_shape=[None, p], name='LSTM1', kernel_initializer=Init, recurrent_initializer=Init),
#   Dropout(.2, name='d1'),
#   LSTM(100, name='LSTM2', kernel_initializer=Init, recurrent_initializer=Init),
#   Dropout(.2, name='d2'),
#   Dense(Ny, name='out', kernel_initializer=Init) ], name='RNN_model') # we build Ny forecasts
# m.summary()
# m.compile(optimizer='adam', loss='mean_squared_error')
# hist = m.fit(taX, taY, epochs=20, batch_size=32)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.72 µs


In [ ]:
# Embargo Split
Nx = 128
Ny_short = 50
step = 5

df = df.dropna().reset_index(drop=True)

print("df length:", len(df))

features = df.drop(columns=['Close']).values
target = df['delta'].values

LtX, LtY = [], []

for i in range(Nx, len(df) - Ny_short, step):
    x_seq = features[i-Nx:i]
    y_seq = target[i:i+Ny_short]

    if len(x_seq) == Nx and len(y_seq) == Ny_short:
        LtX.append(x_seq)
        LtY.append(y_seq)

print("Sequences created:", len(LtX))

if len(LtX) == 0:
    raise ValueError("No sequences created - reduce Nx or Ny_short")

X = np.stack(LtX).astype(np.float32)
Y = np.stack(LtY).astype(np.float32)

print(X.shape, Y.shape)

# EMBARGO TRAIN / VAL SPLIT

embargo = 50  # prevents leakage

split = int(0.8 * len(X))

X_train = X[:split]
Y_train = Y[:split]

X_val = X[split + embargo:]
Y_val = Y[split + embargo:]

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)


# LSTM MODEL
tf.random.set_seed(0)

model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(Nx, X.shape[2])),
    Dropout(0.2),

    LSTM(100),
    Dropout(0.2),

    Dense(Ny_short)
])

model.compile(
    optimizer='adam',
    loss=keras.losses.Huber()
)

model.summary()

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=32
)

# AUTOREGRESSIVE FORECAST
tX_recent = df.iloc[-Nx:].drop(columns=['Close']).values

preds = []
current_input = tX_recent.copy()

steps = len(pY)  # total test steps

for _ in range(steps // Ny_short + 1):

    pred_delta = model.predict(current_input[np.newaxis, ...])[0]
    preds.extend(pred_delta)

    # update input autoregressively
    for d in pred_delta:
        new_row = current_input[-1].copy()

        # update using predicted delta
        new_row[0] += d  # approximate update

        current_input = np.vstack([current_input[1:], new_row])

preds = np.array(preds[:steps])

# Reconstruct Close Price
last_close = df['Close'].iloc[-1]

pred_close = []
current = last_close

for d in preds:
    current = current + d
    pred_close.append(current)

pY['Close'] = pred_close

df length: 499980
Sequences created: 99961
(99961, 128, 20) (99961, 50)
Train shape: (79968, 128, 20)
Val shape: (19943, 128, 20)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128, 100)       │        48,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 100)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50)             │         5,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,850 (522.85 KB)

 Trainable params: 133,850 (522.85 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 37s 13ms/step - loss: 1.4630e-04 - val_loss: 1.7968e-05
Epoch 2/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 7.7519e-07 - val_loss: 1.6199e-05
Epoch 3/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 3.9174e-07 - val_loss: 1.5565e-05
Epoch 4/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 3.0844e-07 - val_loss: 1.5462e-05
Epoch 5/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 2.7440e-07 - val_loss: 1.5417e-05
Epoch 6/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 2.6047e-07 - val_loss: 1.5400e-05
Epoch 7/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 2.5467e-07 - val_loss: 1.5388e-05
Epoch 8/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 31s 13ms/step - loss: 2.5106e-07 - val_loss: 1.5385e-05
Epoch 9/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - loss: 2.4947e-07 - val_loss: 1.5383e-05
Epoch 10/20
2499/2499 ━━━━━━━━━━━━━━━━━━━━ 31s 13ms/step - loss: 2.4875e-07 - val_loss: 1.5382e-05
Epoch 11/20
2499/24

<font color=green><h3><b>$\gamma$. Visualize forecasts</b><h3>

<font color=green>The plot below: the model memorized the prices from recent history (plus some local noise). Still you can use these predictions to visually (i.e. qualitatively) determine whether predictions are meaningful (i.e. have price-like shape) or just noise.

In [ ]:
# tX_recent = tXY.iloc[-Nx:,:]  # most recent history of the coin
# print(f'tX_recent.shape=(Nx,p)={tX_recent.shape}')
# pY['Close'] = m.predict(tX_recent.values[np.newaxis,...]).flatten()  # the model expects a 3-tensor (K=1,Nx,p)
# ax = pd.concat([tX_recent.Close, pY.Close]).plot(figsize=(30,3), title='Most recent input X and forecasted output Y');
# tX_recent.Close.plot(ax=ax, grid=True);
# ax.legend(["Future predicted closing prices", "Historical (train) closing prices"]);

# ── 1. Training loss curve ────────────────────────────────────────────────
# train_loss = history.history["loss"]
# val_loss   = history.history["val_loss"]
# epochs_range = range(1, len(train_loss) + 1)

# fig, axes = plt.subplots(1, 2, figsize=(30, 3))

# axes[0].plot(epochs_range, train_loss, label="Train Huber Loss",      color="steelblue", linewidth=2)
# axes[0].plot(epochs_range, val_loss,   label="Validation Huber Loss", color="darkorange", linewidth=2, linestyle="--")
# axes[0].set_title("Training vs Validation Loss (Huber) over 20 Epochs", fontsize=12)
# axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Huber Loss")
# axes[0].legend(); axes[0].grid(True)

# # ── 2. Historical Close + Forecast ───────────────────────────────────────
# hist_close = df["Close"].iloc[-Nx:].reset_index(drop=True)
# hist_idx   = range(len(hist_close))
# future_idx = range(len(hist_close), len(hist_close) + len(pY))

# axes[1].plot(hist_idx,   hist_close.values,  label=f"Historical Close (last {Nx} steps)", color="steelblue", linewidth=1.5)
# axes[1].plot(future_idx, pY["Close"].values, label=f"LSTM Forecast ({len(pY)} steps)",    color="tomato",    linewidth=1.5, linestyle="--")
# axes[1].axvline(x=len(hist_close) - 1, color="gray", linestyle=":", linewidth=1, label="Forecast start")
# axes[1].set_title("Most Recent Historical Prices and LSTM Multi-Step Forecast", fontsize=12)
# axes[1].set_xlabel("Time Step"); axes[1].set_ylabel("Close Price")
# axes[1].legend(); axes[1].grid(True)

# plt.tight_layout()
# plt.show()

# # ── 3. Summary stats ──────────────────────────────────────────────────────
# print(f"Last observed Close : {df['Close'].iloc[-1]:.4f}")
# print(f"Forecast min / max  : {pY['Close'].min():.4f} / {pY['Close'].max():.4f}")
# print(f"Forecast steps      : {len(pY)}")
# print(f"Final train loss    : {train_loss[-1]:.2e}")
# print(f"Final val   loss    : {val_loss[-1]:.2e}")

<font color=green>

1. The model generates a baseline submission CSV file, see Colab folder (🗀 on the left).
1. You can download the generated CSV file and submit it to Kaggle.

<font color=green><h3><b>$\delta$. Make predictions</b><h3>

In [ ]:
assert len(pY) == len(pY.index), "pY row count mismatch"
assert pY["Close"].isna().sum() == 0, "NaN values found in forecast - check autoregressive loop"

print(f"Saving {len(pY)} predictions to CSV ...")
print(pY.describe().round(4))

ToCSV(pY, '📈Baseline🐍')
print("\n✅ Submission file saved: 📈Baseline🐍.csv")

Saving 24431 predictions to CSV ...
          Close
count  24431.00
mean       1.26
...         ...
75%        1.32
max        1.38

[8 rows x 1 columns]

✅ Submission file saved: 📈Baseline🐍.csv


<font color=green><h3><b>$\epsilon$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title:**

Return-based and rolling-window feature engineering made the crypto sequence more learnable for the LSTM.

2. **Idea Discovery:**

 The raw Close series is highly noisy and non-stationary, so predicting absolute price directly can make the model focus too much on drifting price level instead of short-term movement. Because of that, we transformed the series into change-based and local-context features: delta, log_return, OC_diff, HL_diff, rolling standard deviation, momentum, slope, and a binary direction feature over 5, 10, and 20 step windows. This was motivated by standard forecasting practice, where differencing and transformations help stabilize the series and rolling windows summarize recent behavior.


3. **Finding's Importance:**

 This idea mattered because an LSTM usually learns better from short-run movement, volatility, and local trend than from raw price level alone. In financial time series, absolute prices drift over time, while returns and rolling statistics often capture the structure that is more relevant for near-future prediction. Differencing is widely used to reduce non-stationarity, and time-windowed inputs are a standard way to present sequential structure to neural forecasting models.


4. **Experiment Setup:**

 We started from the baseline feature columns and expanded the preprocessing pipeline with engineered temporal features. After creating differenced and rolling columns, we removed the resulting missing rows and reset the index. We then converted the data into supervised sequences using 128 input timesteps and 50 output timesteps, with the target defined as future delta values instead of future raw Close values. We also used a chronological train/validation split with an embargo gap to reduce temporal leakage between training and validation windows. After forecasting deltas autoregressively, we reconstructed predicted closing prices by cumulatively adding the predicted changes back to the last observed close.


5. **Results:**

 This preprocessing pipeline became the default representation for our LSTM because it gave the model richer temporal structure than the raw baseline inputs alone. In particular, it allowed the network to see recent change, volatility, and short-horizon trend instead of relying only on price level. In our notebook, the practical result was [insert your exact validation-loss or leaderboard improvement here], which was strong enough to justify keeping return-based targets and rolling engineered features in the final pipeline.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: Huber Loss Reduced Sensitivity to Crypto Price Outliers and Stabilized LSTM Training Compared to MSE.

2. **Idea Discovery**: Cryptocurrency price series are characterized by abrupt, high-magnitude spikes and crashes that are fundamentally different from the underlying trend the model should learn. When MSE is used as the loss function, these extreme residuals are squared, causing them to dominate the gradient updates and pulling the model's weights toward memorizing rare events rather than learning the general dynamics of the series. Huber loss addresses this by behaving quadratically (like MSE) for small residuals and linearly (like MAE) for residuals exceeding a threshold δ, thereby down-weighting the influence of large errors during backpropagation. This property is well-suited to financial time series, where sudden market movements occur but should not override the model's ability to forecast typical price behavior. The decision to use Huber loss was motivated by the observation that raw delta values (our target) include occasional large-magnitude jumps, and by prior literature demonstrating that Huber loss significantly improves LSTM resilience to extreme events in stock market forecasting contexts.

3. **Finding's Importance**: Loss function selection directly shapes what an LSTM learns to minimize, and in a noisy financial setting, this choice determines whether the model converges toward robust generalization or toward outlier-driven memorization. By using Huber loss, we allowed the optimizer to focus on the structure of typical price changes while not being heavily penalized for the infrequent but inevitable large moves in crypto markets. The resulting training loss curve confirms stable convergence: train loss declined from 1.56e-4 at epoch 1 to 2.46e-7 at epoch 20, a reduction of more than three orders of magnitude, without divergence or oscillation. This behavior would be harder to achieve with MSE under the same data conditions given crypto's volatility.

4. **Experiment Setup**: We built a two-layer LSTM (100 units per layer, Dropout 0.2 after each layer, Dense output of 50 steps) compiled with the Adam optimizer and Huber loss (TensorFlow default δ = 1.0). The model was trained on 79,968 sequences of 128 input steps predicting 50 future delta steps, with an embargo gap of 50 sequences between the 80% training split and the 20% validation split to prevent temporal leakage. Training ran for 20 epochs with batch size 32. The same architecture with MSE loss was not run due to runtime constraints, so the comparison between Huber and MSE is qualitative and based on the literature rather than a direct in-notebook ablation.

5. **Results**: The model trained with Huber loss achieved a final training loss of 2.46e-7 and a final validation loss of 1.54e-5. The gap between train and validation loss is notable, and likely reflects a combination of two factors. First, Dropout layers are active during training but disabled at inference time, which systematically lowers validation loss relative to training loss for well-regularized models — a known phenomenon in LSTM training. Second, the model was trained on sequences drawn from the earlier 80% of the data, while validation sequences represent a later and potentially distributional distinct market period, a common challenge in financial time series where market regimes shift over time. The forecast visualization confirms that the predicted closing price series starts from 1.1258 (the last observed close), maintains a plausible range of 0.9529–1.1260 over 24,431 future steps, and contains no NaN values, indicating the autoregressive reconstruction pipeline is numerically stable.

<font color=green><h3><b>$\zeta$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Hyndman, R. J., & Athanasopoulos, G. (2021). Transformations and adjustments. In Forecasting: Principles and practice (3rd ed.). OTexts. Retrieved April 11, 2026, from https://otexts.com/fpp3/transformations.html

2. Hyndman, R. J., & Athanasopoulos, G. (2021). Stationarity and differencing. In Forecasting: Principles and practice (3rd ed.). OTexts. Retrieved April 11, 2026, from https://otexts.com/fpp3/stationarity.html
3. TensorFlow. (2024, August 16). Time series forecasting. https://www.tensorflow.org/tutorials/structured_data/time_series
4. Huber, P. J. (1964). Robust estimation of a location parameter. *Annals of Mathematical Statistics*, 35(1), 73–101. https://doi.org/10.1214/aoms/1177703732

5. Chai, K., et al. (2023). Robust stock market forecasting using Huber loss: Mitigating extreme events and outliers in LSTM models. ResearchGate. https://www.researchgate.net/publication/373240182

6. Brownlee, J. (2020). How to diagnose overfitting and underfitting of LSTM models. Machine Learning Mastery. https://machinelearningmastery.com/diagnose-overfitting-underfitting-lstm-models/

<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 774 sec > 300 sec limit!!!


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
Try [**Common Starter Ideas**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=oD-fFbF2ZSBl&line=1&uniqifier=1)

#### **Preprocessing**
1. Try [correlation loss/metric](https://duckduckgo.com/?q=correlation+loss+in+tensorflow&ia=web) or equivalent
1. Try longer/shorter history. FYI: GPU may not fit all observations, but you could lower the precision or simplify DNN
1. Try forecasting returns (differences or log differences at different lags) instead of actual values. Returns might appear "more" stationary (You'll need to compute forecasted prices from forecasted returns later)
1. Try new features: differences, fractions, powers of existing features, lagged features or lagged differences,..
1. Try a different time scale. Eg. forecasting every $k$ steps and then imputing interim values
1. Try technique in HOML pp.509-510
1. Try (programmatically) assigning higher/lower weights to history or historical events (such as extreme events)
1. Try further smoothing/averaging and forecasting values at sparser intervals
1. Try forecasting just the trend line

### **Summarized ideas from past participants**
1. Corr=0.578820583, TA rank=1, SP25:
    1. **Preprocessing:** We experimented with resampling techniques—linear interpolation, spline interpolation, and librosa's resample function—to transform the original close-price time series into fixed-length, uniformly scaled sequences. This approach aimed to reduce noise and volatility, making the data more consistent for LSTM input and improving generalization. They also adopted lag-based features, incorporating the mean and log transforms at several past timesteps (lags 7, 21, and 49), inspired by high-performing solutions. These techniques resulted in significant performance gains, with average Pearson correlation improvements from ~0.17 (no resampling) to ~0.55 (with advanced resampling).
    1. **Modeling:** The primary modeling innovation was placing a 1D convolutional layer before stacked LSTM layers, as recommended in Geron's HOML book. This structure allowed the network to detect local temporal patterns before exploiting long-term dependencies. The model used overlapping data sequences and included dropout layers (increased from 0.2 to 0.5) to reduce overfitting. Adding the convolutional layer and increasing dropout led to a marked boost in correlation from ~0.41 to ~0.57, demonstrating the benefit of hybrid CNN-LSTM architectures for time series prediction.
1. Corr=0.5550498267, TA rank=2, SP25:
    1. **Preprocessing:** We noticed that training on the entire 500k-point dataset resulted in very slow iterations and problematic prediction behavior (vanishing/exploding values). To address this, they sampled approximately 45k points from the data and implemented stride-based sequence extraction, predicting only every 100th point and interpolating the rest. This approach, inspired by literature and instructor suggestions, improved model generalization and reduced overfitting by simplifying the training data and the prediction task.
    1. **Modeling:** After initial modeling failed to capture realistic market behavior, the team post-processed LSTM outputs by applying cubic spline interpolation across stride-delimited predictions. To better mimic the noise and volatility found in actual market data, they then added synthetic market-like noise to the interpolated results. This creative augmentation resulted in a more plausible price trajectory and significantly improved their correlation score on the public leaderboard, demonstrating that post-processing and data realism can compensate for some modeling limitations.
1. Corr=0.4802723167, TA rank=3, SP25:
    1. **Preprocessing:** We reframed the forecasting problem to predict the change (delta) in crypto closing value rather than the actual value, based on the observation that values changed minimally between days. They engineered features such as the difference between Open & Close and High & Low, as well as rolling statistics (standard deviation, slope, momentum) over various windows, the log of yesterday's returns, and the direction of price change. The effectiveness of each feature was validated by monitoring reductions in validation loss, with beneficial features integrated into the final pipeline.
    1. **Modeling:** The modeling strategy shifted to predicting shorter time horizons (e.g., 500 or 1000 time steps) in an autoregressive manner, reducing model complexity and error accumulation compared to predicting the entire sequence at once. A two-layer LSTM model with dropout was trained to predict these shorter sequences, using the model's outputs recursively for full test-set prediction. The introduction of an embargo period between training and validation helped mitigate data leakage due to the autoregressive process, and loss functions like MSE and Huber loss were monitored to guide modeling decisions.
1. Corr=0.6404617884, TA rank=1, FA24:
    1. **Preprocessing:**  The dataset was downsampled by aggregating the original high-frequency data into a larger time scale (such as daily), significantly reducing the number of observations and making model training more manageable. This approach helped mitigate noise and memory issues, prioritizing the capture of price trends over granular precision. Return, rather than closing price, was used as the target variable, since return is more stationary and easier to scale, making model inputs simpler to handle.
    1. **Modeling:** A sliding window strategy was implemented to generate training samples, maximizing the ability to capture sequential patterns regardless of the starting point. Experiments showed that a stride of 1 (fully overlapping windows) yielded the best results and boosted model performance given the small dataset after downsampling. Technical indicators such as Relative Strength Index (RSI) and Exponential Moving Averages (EMA 5 and EMA 10) were added as features to improve trend prediction, resulting in a notable increase in the model's Pearson correlation score. A 1D convolutional neural network was employed for its effectiveness with time series data, further leveraging these preprocessing innovations.
1. Corr=0.6013251159, TA rank=2, FA24:
    1. **Preprocessing:** We engineered an ROI (Return on Investment) feature, calculated as Close minus Open, to take advantage of its greater stationarity compared to raw closing prices. To capture temporal patterns, we introduced lagged features for the Close price (with shifts of 1, 2, and 3), rolling means (window sizes 2 and 3), and encoded the day of the week, handling missing values appropriately. Feature selection was attempted using a Random Forest and permutation importance, but ultimately we chose to retain all features given the lack of improvement from selection.
    1. **Modeling:** We experimented with both deep learning and tree-based models, including RNNs (with LSTM layers) and `HistGradientBoostingRegressor`. Simpler RNN architectures proved effective for the univariate time series, surpassing the baseline, while feature engineering notably boosted performance with gradient boosting. Our final strategy involved predicting ROI instead of the closing price—this shift was key, yielding a dramatic increase in our leaderboard score and demonstrating the importance of target selection in time series forecasting.
1. Corr=0.6404617884, TA rank=3, FA24:
    1. **Preprocessing:** We enhanced our dataset by engineering features specifically designed to capture temporal patterns, such as 5-day and 20-day moving averages and lagged values (1-day and 7-day lags) based on the closing price. Our goal was to help the model “see” both short- and long-term trends as well as immediate past values, which are critical in time series forecasting. To ensure the integrity and consistency of our input, we removed any resulting NaN values, split the data into train and test sets, and normalized features using a StandardScaler.
    1. **Modeling:** We initially experimented with an LSTM-based model to capture sequential dependencies in the data but found it struggled to learn effectively and was computationally intensive. To address these challenges, we switched to a `SimpleRNN`, prioritizing computational efficiency and the ability to capture basic sequential trends without excessive complexity. We also tested different RNN architectures, including adjusting layer sizes and trying GRU units, ultimately finding that GRUs performed better than LSTMs given our preprocessing pipeline. All modeling choices aimed to balance capturing sequence information with efficiency and robust performance.

### **Further resources:**
1. *What optimizer is the best for building time series model using LSTM using Keras? What method should be used to evaluate the performance ...* Quora. [2019](https://www.quora.com/What-optimizer-is-the-best-for-building-time-series-model-using-LSTM-using-Keras-What-method-should-be-used-to-evaluate-the-performance-of-the-different-optimizers)
1. Arthur, R. *A general method for resampling autocorrelated spatial data*. University of Exeter, Department of CS. [1/12/24](https://scholar.google.com/scholar?output=instlink&q=info:yNEqPeixd_QJ:scholar.google.com/&hl=en&as_sdt=0,21&scillfp=9502787437685160241&oi=lle)
1. Avatrade. *Moving Average Forex Strategy Technical Analysis Indicators & Strategies*. [n.d.](https://www.avatrade.com/education/technical-analysis-indicators-strategies/moving-average-forex-strategy)
1. Brownlee, J. *How to Develop LSTM Models for Time Series Forecasting*. ML Mastery. [11/13/18](https://machinelearningmastery.com/how-to-develop-lstm-models-for-time-series-forecasting)
1. Essa, H. M., & Murshid, A. M. *Optimizing Image Processing with CNNs through Transfer Learning: Survey*. Al-Kitab Journal for Pure Sciences, 7(1), 57-68. [2023](https://isnra.net/index.php/kjps/article/view/926)
1. Hong, Z. *Predicting stock returns: A guide to feature engineering for Financial Data*. Medium. [12/27/24](https://medium.com/@zhonghong9998/)predicting-stock-returns-a-guide-to-feature-engineering-for-financial-data-bbf6700b11d7
1. Kamalov, F., et al. *Financial Forecasting with ML: Price Vs Return*. Journal of CS, 17(3), 251-264. [2021](https://papers.ssrn.com/sol3/Delivery.cfm?abstractid=3808539)
1. Lioudis, N. *Why is the forex market open 24 hours a day?* Investopedia. [9/17/24](https://www.investopedia.com/ask/answers/how-forex-market-trade-24-hours-day)
1. Oanda smarter trading. *How to use moving averages*. [n.d.](https://www.oanda.com/bvi-en/cfds/learn/indicators-oscillators/filtering-out-the-noise-moving-averages)
1. Patil, Malini. *Interpolation Techniques in Image Resampling*. International Journal of Engineering and Technology (IJET). 7. 567-570. [2018](https://www.researchgate.net/profile/Manjunath-Shivanandappa/publication/357670987_Interpolation_Techniques_in_Image_Resampling/links/61d9031fd45006081696c4b5/Interpolation-Techniques-in-Image-Resampling.pdf)
1. Singh, S., et al. Prediction of bitcoin stock price using feature subset optimization. [4/15/24](https://www.sciencedirect.com/science/article/pii/S2405844024044463)
1. SKL Authors. *Lagged features for time series forecasting*. Scikit-learn. [2025](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html)
1. Smith, L. N. *A disciplined approach to NN hyper-parameters: Part 1--learning rate, batch size, momentum, and weight decay*. arXiv [2018](https://arxiv.org/abs/1803.09820)
1. SPSS Modeler Subscription. Ibm.com. [8/17/21](https://www.ibm.com/docs/sl/spss-modeler/saas?topic=data-characteristics-time-series)
1. Sugghi. *Training 3rd place solution (Code notebook)*. Kaggle. G-Research Crypto Forecasting Competition. [3/9/24](https://www.kaggle.com/code/sugghi/training-3rd-place-solution)
1. Sugghi. Training 3rd place solution. Kaggle. [5/7/22](https://www.kaggle.com/code/sugghi/training-3rd-place-solution)
1. Suh, Y., et al. *Stochastic class-based hard example mining for deep metric learning*. IEEE/CVF CVPR. pp.7251-7259. [2019](https://openaccess.thecvf.com/content_CVPR_2019/html/Suh_Stochastic_Class-Based_Hard_Example_Mining_for_Deep_Metric_Learning_CVPR_2019_paper.html)
1. Svolba, G. *Determining the best length of the history of your timeseries data for timeseries forecasting*. Medium. [3/10/22](https://gerhard-svolba.medium.com/determining-the-best-length-of-the-history-of-your-timeseries-data-for-timeseries-forecasting-f8600a3c086)
1. Talaviya, A. *Time-series analysis and stock price forecast modeling: All you need to know*. Medium. [11/14/22](https://medium.com/@avikumart_/time-series-analysis-and-stock-price-forecast-modeling-all-you-need-to-know-fe66c06e50ae)
1. TF authors. *Time Series forecasting:  TF Core*. TF. [n.d.](https://www.tensorflow.org/tutorials/structured_data/time_series#advanced_autoregressive_model)
1. Timescale. *Time-Series Forecasting: Definition, Methods, and Applications*. [11/6/24](https://www.timescale.com/blog/what-is-time-series-forecasting/#neural-networks)
1. Token Metrics Team. *10 Best Indicators for Crypto Trading and Analysis in 2024*. [2024](https://www.tokenmetrics.com/blog/best-indicators-for-crypto-trading-and-analysis#10-on-chain-metrics)
1. Winastwan, R. *ML forecasting of time series*. Train in Data's Blog. [3/23/24](https://www.blog.trainindata.com/machine-learning-forecasting)
1. Yang, H., & Liu, W. *Visual Interpretability and Comparison of Latent Features in Deep CNNs. IEEE Transactions on NN and Learning Systems. [2023](https://link.springer.com/article/10.1631/FITEE.1700808)
1. Yu, H., & Chen, H. *LSTM-based approach for predicting changes in cryptocurrency price*. IEEE ICCCN. pp. 895-899. [2019](https://doi.org/10.1109/ICCC47050.2019.9064155)

</div> </details>
